# 01 - EDA & Preprocessing
## Studi Komparatif Multilingual Sentence-BERT dan IndoBERT untuk Deteksi Plagiarisme Semantik

Dataset:
- Quora Paraphrase Indonesia (Kaggle)
- MSRP Indonesia (HuggingFace)

In [ ]:
# Install dependencies
!pip install -q pandas numpy matplotlib scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os

os.makedirs('data', exist_ok=True)
os.makedirs('assets', exist_ok=True)

## Download Dataset MSRP Indonesia (HuggingFace)

In [ ]:
# Download langsung dari GitHub raw (sumber asli HuggingFace)
train_url = 'https://media.githubusercontent.com/media/jakartaresearch/hf-datasets/main/msrp/id_train.csv'
val_url = 'https://media.githubusercontent.com/media/jakartaresearch/hf-datasets/main/msrp/id_test.csv'

msrp_train = pd.read_csv(train_url)
msrp_val = pd.read_csv(val_url)

print(f"Train: {len(msrp_train)} rows")
print(f"Validation: {len(msrp_val)} rows")
print(f"Columns: {msrp_train.columns.tolist()}")
print(f"\nLabel distribution (train):")
print(msrp_train['label'].value_counts())

# Simpan ke CSV lokal
msrp_train.to_csv('data/id_msrp_train.csv', index=False)
msrp_val.to_csv('data/id_msrp_val.csv', index=False)
print("\nSaved: data/id_msrp_train.csv")
print("Saved: data/id_msrp_val.csv")

## Download Dataset Quora Indonesia (Kaggle)

**Opsi 1**: Download manual dari Kaggle, upload ke Colab

**Opsi 2**: Pakai Kaggle API (butuh kaggle.json)

In [ ]:
# Uncomment jika punya kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d louisowen6/quora-paraphrasing-bahasa-indonesia-version
# !unzip -q quora-paraphrasing-bahasa-indonesia-version.zip -d data/

# Jika sudah punya file JSON:
# with open('data/quora_indo_train.json', 'r') as f:
#     quora_train = json.load(f)
# df_quora_train = pd.DataFrame(quora_train)
# print(f"Quora Train: {len(df_quora_train)} rows")

## EDA - Distribusi Label

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

msrp_train['label'].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('MSRP Train - Label Distribution')
axes[0].set_xlabel('Label (0=Bukan Plagiarisme, 1=Plagiarisme)')
axes[0].set_ylabel('Count')

msrp_val['label'].value_counts().plot(kind='bar', ax=axes[1], color=['skyblue', 'salmon'])
axes[1].set_title('MSRP Validation - Label Distribution')
axes[1].set_xlabel('Label (0=Bukan Plagiarisme, 1=Plagiarisme)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('assets/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Contoh Pasangan Teks

In [ ]:
print("=== CONTOH PLAGIARISME SEMANTIK (Label=1) ===")
pos = msrp_train[msrp_train['label'] == 1].head(3)
for _, row in pos.iterrows():
    print(f"Teks 1: {row['sentence1']}")
    print(f"Teks 2: {row['sentence2']}")
    print("---")

print("\n=== CONTOH BUKAN PLAGIARISME (Label=0) ===")
neg = msrp_train[msrp_train['label'] == 0].head(3)
for _, row in neg.iterrows():
    print(f"Teks 1: {row['sentence1']}")
    print(f"Teks 2: {row['sentence2']}")
    print("---")

## Preprocessing

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
msrp_train['sentence1_clean'] = msrp_train['sentence1'].apply(clean_text)
msrp_train['sentence2_clean'] = msrp_train['sentence2'].apply(clean_text)
msrp_val['sentence1_clean'] = msrp_val['sentence1'].apply(clean_text)
msrp_val['sentence2_clean'] = msrp_val['sentence2'].apply(clean_text)

print("Contoh setelah cleaning:")
print(f"Original: {msrp_train.iloc[0]['sentence1']}")
print(f"Cleaned:  {msrp_train.iloc[0]['sentence1_clean']}")

## Simpan Data Clean

In [ ]:
msrp_train[['sentence1_clean', 'sentence2_clean', 'label']].to_csv('data/msrp_train_clean.csv', index=False)
msrp_val[['sentence1_clean', 'sentence2_clean', 'label']].to_csv('data/msrp_val_clean.csv', index=False)

print(f"Saved: data/msrp_train_clean.csv ({len(msrp_train)} rows)")
print(f"Saved: data/msrp_val_clean.csv ({len(msrp_val)} rows)")